In [2]:
import pandas as pd
import re
import csv
import pytesseract
from tempfile import TemporaryDirectory
from PIL import Image
from pathlib import Path 
from pdf2image import convert_from_path


home = Path()/".."
pdf = home/"data/bini_dict.pdf"


In [33]:
def read_pdf2img(pdf_path):
    """
    Converts the PDF to images and writes the OCR output to a single text file.
    """
    output_folder = home/"data"
    output_folder.mkdir(exist_ok=True)

    with TemporaryDirectory() as tmp:
        storage = Path(tmp)
        convert_from_path(
            pdf_path=Path(pdf_path),
            output_folder=storage,
            fmt='png',
            single_file=False,
            first_page=19, 
        )

        with open(output_folder/"bini_output.txt", "w") as f:
            for x in sorted(storage.glob("*.png")): 
                text = pytesseract.image_to_string(Image.open(x), lang='bini') # after training the model, use lang=['eng', 'bini'] as an argument
                f.write(text + "\n")  # Add newlines between pages
                # fix: pages of the file ebing written get scattered, find out why


read_pdf2img(pdf)

In [ ]:
images = Path("../pdf_img")
output_dir = Path("../output_folder")

for x in images.iterdir():
    name = x.stem
    img = Image.open(x)
    content = pytesseract.image_to_string(img, lang="bini")
    with open(f"{output_dir}/{name}.txt", "w") as f:
        f.write(content)


In [34]:
input_txt   = "../data/output_file.txt"
bini_txt    = "../data/bini_output.txt"
bini_output = "../data/proper_bini_ouput.txt"
bini_csv    = "../data/proper_bini_output.csv"
output_txt  = "../data/bini_words_definitions.txt"
output_csv  = "../data/bini_words_definitions.csv"

with open(bini_txt, "r", encoding="utf-8") as f:
    text = f.read()

# removing unneeded things from the corpus
# text = re.sub(r"\BINI DICTIONARY", "", text)  # remove BINI DICTIONARY
# text = re.sub(r"\2|\4|\6|\7|\9", "", text) # remove certain numbers
# text = re.sub(r"\»", "", text)
# text = re.sub(r"\™", "", text)

# # ---substituting certain values for mistaken counterparts
# text = re.sub(r"\3", "ε", text)
# text = re.sub(r"\1", "i", text)
# # text = re.sub(r"\5", "", text)
# text = re.sub(r"\8", "", text)
# text = re.sub(r"\¢", "c", text)
# text = re.sub(r"\¥", "Y", text)
# # text = re.sub(r"\#", "", text)
# text = re.sub(r"\€", "ε", text)
# text = re.sub(r"\©", "c", text)

# Pattern: word at start of line, then phonetic, then definition
pattern = re.compile(
    r"(?m)^([a-zA-Zɣεↄɽῦυ]+)"           # word
    r"\s+\[.*?\]"                 # phonetic (skip)
    r"\s+(.*?)(?=(?:\n[A-Za-zɣεↄɽῦυ]+ \[)|\Z)",  # definition until next entry or end
    re.S
)

with open(bini_output, "w", encoding="utf-8") as out:
    for match in pattern.finditer(text):
        word = match.group(1).strip()
        definition = match.group(2).replace('\n', ' ').strip()
        out.write(f"{word}: {definition}\n")

with open(bini_output, "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(bini_csv, "w", encoding="utf-8", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["word", "definition"])
    for line in lines:
        if ": " in line:
            word, definition = line.strip().split(": ", 1)
            writer.writerow([word, definition])


In [3]:
bini_output = "../data/proper_bini_ouput.txt"
bini_csv    = "../data/proper_bini_output.csv"

df = pd.read_csv(bini_csv)

df.sort_values(by='word')

,word,definition
71,AQwehi,a title used in ad- dressing the dba. axik-ↄd...
13,AbigεgQε,"abↄ [..] (« I branch, in ab-erha [...] branch ..."
688,Abigεgε,a kind of sedge growing on river banks; the le...
20,Adabi,a deity supposed to stand on the boundary betw...
29,Adbbb,"name of an dba, father of Dba OvHrfave [...]; ..."
...,...,...
1257,ↄwεde,plantain-peel; ikp- akp-eh# [...] scales of fi...
2247,ↄxa,white ants; termites.
2246,ↄxa,"the cotton ftree, Cecwba pentandra; the seeds ..."
2249,ↄxad,"porcupine ( ); ""“hedge- hog""; ↄaε rhiεrhi-tinw..."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2978 entries, 0 to 2977
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   word        2978 non-null   object
 1   definition  2978 non-null   object
dtypes: object(2)
memory usage: 46.7+ KB


In [18]:
# Clean the 'definition' column, keeping only specified symbols and spaces
df_cleaned = df.copy()
df_cleaned['word'] = df_cleaned['word'].apply(lambda x: str.lower(x))
df_cleaned['definition'] = df_cleaned['definition'].apply(lambda x: ' '.join(re.findall(r'[A-Za-zɣεↄɽῦυ]+', str(x))))
# df_cleaned['definition'] = df_cleaned['definition'].apply(lambda x: ' '.replace(r"cf|Yor|Eng|Ibo", ""))

df_cleaned.to_csv("../data/bini_corpus.csv")